# Three fourth-order models over 200 cycles: calculation

Compare BM4Implicit, two-stage Gauss–Legendre4 and classical RK4 at 200 complete steps per cycle. This requested subset and finer step intentionally specialize the standard five-method protocol. Recompute DOP853 with at most 0.01 time units per step (100 steps per cycle) and reuse only the stored Radau audit from `reference_results.csv`.


## Reproducible long-time experiment

The interval [0, 200] contains 200 normalized cycles. Integration uses step 0.005 (40,000 complete steps). Save every four steps at the existing reference times, spaced by 0.02 (10,001 states), without interpolation. All per-step Newton and projection diagnostics are retained in CSV substep columns. The explicitly time-dependent physical Hamiltonian is compared with its reference history, not tested for conservation.


In [1]:
from contextlib import redirect_stderr
from pathlib import Path
import sys

from IPython.display import Markdown, display
import numpy as np

from diagnostics import write_five_method_comparison_csv, load_five_method_comparison_csv
from diagnostics.paths import find_project_root
from potential import GC2DH5Metadata, load_gc2d_h5_potential
from studies import (
    FiveMethodComparisonConfig,
    domain_center,
    latin_hypercube_gc_configuration_with_near_center,
    run_five_method_comparison,
)

In [2]:
# Measured, nondimensionalized GC2D potential used by the project.
project_root = find_project_root(Path.cwd())
notebook_directory = (
    project_root
    / "notebooks/developements/energy/compare_three_order4_models_bm4_gauss_legendre4_rk4_200_steps_per_cycle"
)
notebook_directory.mkdir(parents=True, exist_ok=True)
results_path = notebook_directory / "results.csv"
execution_log_path = notebook_directory / "calculation.log"
data_path = project_root / "data/potential/V1/PHI_2.h5"
magnetic_field = 1.5
characteristic_length = 0.06
mode_selection = (0, 1)
interpolation_order = 3
if not data_path.is_file():
    raise FileNotFoundError(f"Measured HDF5 potential not found: {data_path}")

# Physical parameters and three spatially distributed initial trajectories.
rho = 0.3
coupling_frequency = float(np.pi / 8.0)
trajectory_count = 3
initial_condition_seed = 20260905
domain_margin_fraction = 0.05
near_center_offset_fraction = (0.08, -0.06)

# Two hundred complete steps per normalized cycle; retain the reference grid.
cycle_duration = 1.0
steps_per_cycle = 200
t_span = (0.0, 200.0)
integration_step = cycle_duration / steps_per_cycle
save_interval = 0.02

# Common Newton controls.
newton_absolute_tolerance = 1e-13
newton_relative_tolerance = 1e-12
newton_max_iterations = 40
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

# Recomputed DOP853 reference and reused independent Radau audit.
reference_relative_tolerance = 1e-10
reference_absolute_tolerance = 1e-12
reference_maximum_step = 0.01
audit_relative_tolerance = 1e-11
audit_absolute_tolerance = 1e-13
audit_maximum_step = 0.0125
timing_warmups = 0
timing_repeats = 3

potential = load_gc2d_h5_potential(
    data_path,
    B=magnetic_field,
    characteristic_length=characteristic_length,
    indx=mode_selection,
    interpolation_order=interpolation_order,
)
potential_metadata = potential.metadata
assert isinstance(potential_metadata, GC2DH5Metadata)
initial_configuration = latin_hypercube_gc_configuration_with_near_center(
    potential,
    particle_count=trajectory_count,
    seed=initial_condition_seed,
    center_offset_fraction=near_center_offset_fraction,
    domain_margin_fraction=domain_margin_fraction,
)
config = FiveMethodComparisonConfig(
    rho=rho,
    coupling_frequency=coupling_frequency,
    t_span=t_span,
    integration_step=integration_step,
    save_interval=save_interval,
    absolute_tolerance=newton_absolute_tolerance,
    relative_tolerance=newton_relative_tolerance,
    max_iterations=newton_max_iterations,
    jacobian_relative_step=jacobian_relative_step,
    reference_relative_tolerance=reference_relative_tolerance,
    reference_absolute_tolerance=reference_absolute_tolerance,
    reference_maximum_step=reference_maximum_step,
    audit_relative_tolerance=audit_relative_tolerance,
    audit_absolute_tolerance=audit_absolute_tolerance,
    audit_maximum_step=audit_maximum_step,
    timing_warmups=timing_warmups,
    timing_repeats=timing_repeats,
    distance_convention="periodic",
    progress=True,
)

assert config.step_count == 40_000
assert config.output_sample_count == 10_001
assert initial_configuration.initial_state is not None
assert initial_configuration.initial_state.size == 2 * trajectory_count
near_center_coordinates = (
    np.asarray(domain_center(potential), dtype=float)
    + np.asarray(near_center_offset_fraction) * potential.grid.period
)
np.testing.assert_allclose(
    (initial_configuration.initial_state[0], initial_configuration.initial_state[trajectory_count]),
    near_center_coordinates,
)

display(Markdown(
    f"**Resolved grid:** `{config.step_count}` effective steps of `{integration_step:g}`, "
    f"`{config.output_sample_count}` saved states, `{steps_per_cycle}` steps per cycle, "
    f"and `{trajectory_count}` trajectories on `[{t_span[0]:g}, {t_span[1]:g}]`.\n\n"
    f"**Measured potential:** `{potential_metadata.source_path}`, grid `{potential.grid.shape[0]} x {potential.grid.shape[1]}`, "
    f"selected source fields `{potential_metadata.source_field_indices.tolist()}` and normalized frequencies `{potential.frequencies.tolist()}`."
))
method_names = ("BM4Implicit", "GaussLegendre4", "RK4")
implicit_method_names = ("BM4Implicit", "GaussLegendre4")
reference_path = notebook_directory / "reference_results.csv"
stored_reference = load_five_method_comparison_csv(reference_path)
# Fail before timing if the saved reference belongs to a different experiment.
reference_config = stored_reference.metadata["config"]
for key in ("rho", "coupling_frequency", "t_span", "save_interval", "distance_convention",
            "reference_relative_tolerance", "reference_absolute_tolerance",
            "audit_relative_tolerance", "audit_absolute_tolerance", "audit_maximum_step"):
    expected = getattr(config, key)
    actual = reference_config[key]
    assert (tuple(actual) == expected if isinstance(expected, tuple) else actual == expected), key
assert stored_reference.metadata["experiment"]["potential"] == {
    "source_path": str(data_path.relative_to(project_root)),
    "magnetic_field": magnetic_field, "characteristic_length": characteristic_length,
    "mode_selection": list(mode_selection), "interpolation_order": interpolation_order,
}
np.testing.assert_array_equal(stored_reference.reference.states[:, 0], initial_configuration.initial_state)


**Resolved grid:** `40000` effective steps of `0.005`, `10001` saved states, `200` steps per cycle, and `3` trajectories on `[0, 200]`.

**Measured potential:** `/home/ubuntu/GC2D_intranet/data/potential/V1/PHI_2.h5`, grid `256 x 256`, selected source fields `[15]` and normalized frequencies `[1.0]`.

## Aligned integrations with a reused reference

Only BM4Implicit and Gauss–Legendre4 use Newton; RK4 is explicit. The three model campaigns run concurrently, with three full integrations performed sequentially inside each model worker. All three trajectories advance together. Progress is mirrored to `calculation.log`. Missing or incompatible reference data cause an error; there is no adaptive-reference fallback.


In [3]:
class ExecutionLogTee:
    """Mirror live stderr output to the notebook and a plain-text log file."""

    def __init__(self, notebook_stream, file_stream):
        self.notebook_stream = notebook_stream
        self.file_stream = file_stream

    def write(self, text):
        self.notebook_stream.write(text)
        # Carriage-return progress bars become readable append-only log lines.
        self.file_stream.write(text.replace("\r", "\n"))
        self.file_stream.flush()
        return len(text)

    def flush(self):
        self.notebook_stream.flush()
        self.file_stream.flush()


print(f"Live calculation log: {execution_log_path.relative_to(project_root)}", flush=True)
print(
    "The cell output shows method/repeat events, campaign ETA, and step progress. "
    "From another shell, follow the same stream with "
    f"`tail -f {execution_log_path.relative_to(project_root)}`.",
    flush=True,
)
with execution_log_path.open("w", encoding="utf-8") as log_stream:
    with redirect_stderr(ExecutionLogTee(sys.stderr, log_stream)):
        result = run_five_method_comparison(
            potential,
            initial_configuration,
            config=config,
            method_names=method_names,
            reused_audit_reference=stored_reference.reference,
            parallel_models=True,
        )

gauss_diagnostics = result.solutions["GaussLegendre4"].diagnostics
bm4_diagnostics = result.solutions["BM4Implicit"].diagnostics
rk4_diagnostics = result.solutions["RK4"].diagnostics
assert bm4_diagnostics["projection_solver_formulation"] == "bm4_implicit_reduced"
assert gauss_diagnostics["stage_count"] == 2
assert all(
    result.solutions[name].diagnostics["nonlinear_solver"] == "newton"
    for name in implicit_method_names
)
assert "nonlinear_solver" not in rk4_diagnostics
assert all(
    result.solutions[name].diagnostics["step_count"] == config.step_count
    for name in method_names
)

display(Markdown(
    f"Study completed in **{result.total_study_runtime_seconds:.3f} s** using saved references "
    f"and timed runs. The DOP853/Radau space-time RMS reference discrepancy is "
    f"**{result.reference.time_integrated_rms_floor:.3e}**."
))


Live calculation log: notebooks/developements/energy/compare_three_order4_models_bm4_gauss_legendre4_rk4_200_steps_per_cycle/calculation.log


The cell output shows method/repeat events, campaign ETA, and step progress. From another shell, follow the same stream with `tail -f notebooks/developements/energy/compare_three_order4_models_bm4_gauss_legendre4_rk4_200_steps_per_cycle/calculation.log`.


[five-method study] Recomputing DOP853 with maximum step 0.01; reusing the saved Radau audit.


[five-method study] Running 3 model campaigns in parallel.


[five-method study] Starting timing 1/3: Single-projection implicit BM4; 3 trajectories together, 40000 steps.[five-method study] Starting timing 1/3: Gauss--Legendre (2 stages, order 4); 3 trajectories together, 40000 steps.
[five-method study] Starting timing 1/3: Classical explicit RK4; 3 trajectories together, 40000 steps.


...

BM4Implicit [=============================>]  99.0% (39600/40000, t=198)

BM4Implicit [==============================] 100.0% (40000/40000, t=200)

[five-method study] Completed timing 3/3: Single-projection implicit BM4 in 3534.84 s; campaign 9/9 (100.0%), elapsed 12347.8 s, ETA 0.0 s.


[five-method study] Completed study in 13166.35 s.


Study completed in **13166.346 s** using saved references and timed runs. The DOP853/Radau space-time RMS reference discrepancy is **2.124e+00**.

## Persist the complete calculation

The CSV is deliberately colocated with the calculation and visualization notebooks. Re-running this cell atomically replaces the previous artifact only after the new file has been written successfully.

In [4]:
written_path = write_five_method_comparison_csv(
    result,
    results_path,
    metadata={
        "study_name": "Three fourth-order models over 200 cycles",
        "potential": {
            "source_path": str(data_path.relative_to(project_root)),
            "magnetic_field": magnetic_field,
            "characteristic_length": characteristic_length,
            "mode_selection": mode_selection,
            "interpolation_order": interpolation_order,
        },
        "initial_conditions": {
            "trajectory_count": trajectory_count,
            "seed": initial_condition_seed,
            "domain_margin_fraction": domain_margin_fraction,
            "near_center_offset_fraction": near_center_offset_fraction,
        },
        "reference_reuse": {"path": reference_path.name, "radau_reused": True, "dop853_recomputed": True, "interpolated": False},
        "steps_per_cycle": steps_per_cycle,
        "parallel_models": True,
        "method_names": method_names,
    },
    overwrite=True,
)
assert written_path.is_file()
display(Markdown(
    f"Wrote **{written_path.relative_to(project_root)}** "
    f"({written_path.stat().st_size / 1024**2:.2f} MiB)."
))

Wrote **notebooks/developements/energy/compare_three_order4_models_bm4_gauss_legendre4_rk4_200_steps_per_cycle/results.csv** (16.91 MiB).